In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score, f1_score, confusion_matrix, precision_score, recall_score, classification_report,roc_auc_score,roc_curve,RocCurveDisplay
import os
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from tqdm import tqdm
os.chdir('/home/pgcp-ai/MachineLearning/Datasets/')

In [2]:
sonar = pd.read_csv("Sonar.csv")
sonar

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V52,V53,V54,V55,V56,V57,V58,V59,V60,Class
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032,R
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044,R
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078,R
3,0.0100,0.0171,0.0623,0.0205,0.0205,0.0368,0.1098,0.1276,0.0598,0.1264,...,0.0121,0.0036,0.0150,0.0085,0.0073,0.0050,0.0044,0.0040,0.0117,R
4,0.0762,0.0666,0.0481,0.0394,0.0590,0.0649,0.1209,0.2467,0.3564,0.4459,...,0.0031,0.0054,0.0105,0.0110,0.0015,0.0072,0.0048,0.0107,0.0094,R
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203,0.0187,0.0346,0.0168,0.0177,0.0393,0.1630,0.2028,0.1694,0.2328,0.2684,...,0.0116,0.0098,0.0199,0.0033,0.0101,0.0065,0.0115,0.0193,0.0157,M
204,0.0323,0.0101,0.0298,0.0564,0.0760,0.0958,0.0990,0.1018,0.1030,0.2154,...,0.0061,0.0093,0.0135,0.0063,0.0063,0.0034,0.0032,0.0062,0.0067,M
205,0.0522,0.0437,0.0180,0.0292,0.0351,0.1171,0.1257,0.1178,0.1258,0.2529,...,0.0160,0.0029,0.0051,0.0062,0.0089,0.0140,0.0138,0.0077,0.0031,M
206,0.0303,0.0353,0.0490,0.0608,0.0167,0.1354,0.1465,0.1123,0.1945,0.2354,...,0.0086,0.0046,0.0126,0.0036,0.0035,0.0034,0.0079,0.0036,0.0048,M


In [3]:
le = LabelEncoder()
sonar["Class"] = le.fit_transform(sonar["Class"])

In [4]:
X, y = sonar.drop("Class", axis = 1), sonar["Class"]

In [5]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,stratify=y,random_state=26)

In [6]:
svm = SVC(kernel='linear')
svm.fit(X_train,y_train)
y_pred = svm.predict(X_test)
accuracy_score(y_test,y_pred)

0.8412698412698413

In [7]:
Cs = np.linspace(0.001,5,20)
scores = []
for c in tqdm(Cs):
    svm = SVC(kernel='linear',C=c,probability=True,random_state=26)
    svm.fit(X_train,y_train)
    y_pred = svm.predict(X_test)
    acc = accuracy_score(y_test,y_pred)
    y_pred_prob = svm.predict_proba(X_test)
    log_loss_score = log_loss(y_test,y_pred_prob)
    scores.append([c,acc,log_loss_score])

100%|███████████████████████████████████████████| 20/20 [00:00<00:00, 49.20it/s]


In [8]:
df_scores = pd.DataFrame(scores,columns=['C','Accuracy Score','Log Loss'])
df_scores.sort_values('Log Loss',ascending=True)

,C,Accuracy Score,Log Loss
3,0.790316,0.841270,0.488054
4,1.053421,0.841270,0.493004
8,2.105842,0.809524,0.495309
7,1.842737,0.825397,0.495447
6,1.579632,0.841270,0.496055
9,2.368947,0.809524,0.497287
5,1.316526,0.841270,0.497609
2,0.527211,0.809524,0.498198
10,2.632053,0.809524,0.500052
1,0.264105,0.793651,0.504347


### To use `predict_proba` in SVC we have to first put `probability=True` and re-assign `random_state=26` in SVC Constructor to enable `predict_proba` outputs and avoid errors

In [9]:
svm = SVC(C=0.790316,kernel='linear',probability=True,random_state=26)
svm.fit(X_train,y_train)
y_pred_prob = svm.predict_proba(X_test)
log_loss(y_test,y_pred_prob)

0.48807796683201926

### Polynomial Kernel

In [11]:
svm = SVC(C=0.790316,kernel='poly', degree=5,probability=True,random_state=26)
svm.fit(X_train,y_train)
y_pred_prob = svm.predict_proba(X_test)
log_loss(y_test,y_pred_prob)

0.3911085710622369

In [21]:
Cs = np.linspace(0.001,5,20)
degree = [2,3,4]
scores = []
for c in tqdm(Cs):
    for d in degree: 
        svm = SVC(kernel='poly',C=c,probability=True,degree=d,random_state=26)
        svm.fit(X_train,y_train)
        y_pred = svm.predict(X_test)
        acc = accuracy_score(y_test,y_pred)
        y_pred_prob = svm.predict_proba(X_test)
        log_loss_score = log_loss(y_test,y_pred_prob)
        scores.append([c,d,acc,log_loss_score])

100%|███████████████████████████████████████████| 20/20 [00:02<00:00,  8.25it/s]


In [19]:
df_scores = pd.DataFrame(scores,columns=['C','Degree','Accuracy Score','Log Loss'])
df_scores.sort_values(['Log Loss','Accuracy Score'],ascending=[True,False])

,C,Degree,Accuracy Score,Log Loss
59,5.000000,4,0.809524,0.390880
56,4.736895,4,0.809524,0.390901
53,4.473789,4,0.809524,0.391032
50,4.210684,4,0.809524,0.391403
41,3.421368,4,0.825397,0.391449
47,3.947579,4,0.825397,0.391599
44,3.684474,4,0.825397,0.391740
38,3.158263,4,0.825397,0.393758
35,2.895158,4,0.825397,0.395712
32,2.632053,4,0.825397,0.398660


### Radial Kernel

In [22]:
Cs = np.linspace(0.001,5,20)
Gs = np.linspace(0.001, 5, 20)
scores = []
for c in tqdm(Cs):
    for g in Gs: 
        svm = SVC(kernel='rbf',C=c,probability=True,gamma = g,random_state=26)
        svm.fit(X_train,y_train)
        y_pred = svm.predict(X_test)
        acc = accuracy_score(y_test,y_pred)
        y_pred_prob = svm.predict_proba(X_test)
        log_loss_score = log_loss(y_test,y_pred_prob)
        scores.append([c,g,acc,log_loss_score])

100%|███████████████████████████████████████████| 20/20 [00:08<00:00,  2.40it/s]


In [23]:
df_scores = pd.DataFrame(scores, columns = ['C', 'G', 'Accuracy Score', 'Log Loss'])

In [24]:
df_scores.sort_values(['Log Loss', 'Accuracy Score'], ascending = [True, False])

,C,G,Accuracy Score,Log Loss
286,3.684474,1.579632,0.904762,0.231548
306,3.947579,1.579632,0.904762,0.231668
326,4.210684,1.579632,0.904762,0.231739
266,3.421368,1.579632,0.904762,0.231763
346,4.473789,1.579632,0.904762,0.231820
...,...,...,...,...
280,3.684474,0.001000,0.539683,0.698926
260,3.421368,0.001000,0.539683,0.699131
200,2.632053,0.001000,0.539683,0.699137
220,2.895158,0.001000,0.539683,0.699229


### Sigmoid Kernel

In [26]:
Cs = np.linspace(0.001,5,20)
coef = np.linspace(-5, 5, 20)
scores = []
for c in tqdm(Cs):
    for cf in coef: 
        svm = SVC(kernel='sigmoid',C=c,probability=True,coef0=cf,random_state=26)
        svm.fit(X_train,y_train)
        y_pred = svm.predict(X_test)
        acc = accuracy_score(y_test,y_pred)
        y_pred_prob = svm.predict_proba(X_test)
        log_loss_score = log_loss(y_test,y_pred_prob)
        scores.append([c,cf,acc,log_loss_score])


100%|███████████████████████████████████████████| 20/20 [00:08<00:00,  2.33it/s]


In [27]:
df_scores = pd.DataFrame(scores, columns = ['C', 'coef0', 'Accuracy', 'Log Loss'])
df_scores.sort_values(['Log Loss', 'Accuracy'], ascending = [True, False])

,C,coef0,Accuracy,Log Loss
384,5.000000,-2.894737,0.857143,0.420820
364,4.736895,-2.894737,0.857143,0.421167
344,4.473789,-2.894737,0.857143,0.422775
304,3.947579,-2.894737,0.873016,0.422984
284,3.684474,-2.894737,0.873016,0.423297
...,...,...,...,...
209,2.632053,-0.263158,0.539683,0.731457
310,3.947579,0.263158,0.571429,0.732252
148,1.842737,-0.789474,0.698413,0.737013
189,2.368947,-0.263158,0.555556,0.739140
